In [2]:
#PBKDF2-Based Version - research-grade security
import hashlib
import os
import base64
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend


# ---------------- Cipher Logic ---------------- #
def encode_text(input_text):
    input_text = input_text.lower()
    mapping = "☺☻♥♦♣♠•◘○◙♂♀♪♫☼►◄↕‼¶§▬↨↑↓→"
    cipher_text = ""

    for ch in input_text:
        if 'a' <= ch <= 'z':
            cipher_text += mapping[ord(ch) - ord('a')]
        else:
            cipher_text += ch

    return cipher_text


def decode_text(cipher_text):
    mapping = "☺☻♥♦♣♠•◘○◙♂♀♪♫☼►◄↕‼¶§▬↨↑↓→"
    decoded_text = ""

    for ch in cipher_text:
        if ch in mapping:
            decoded_text += chr(mapping.index(ch) + ord('a'))
        else:
            decoded_text += ch

    return decoded_text


# ---------------- PBKDF2 Key Derivation ---------------- #
def derive_aes_key(password, pin, salt=None):
    """
    Derives AES-256 key using PBKDF2-HMAC (SHA-256)
    """
    combined = (password + pin).encode()

    if salt is None:
        salt = os.urandom(16)

    key = hashlib.pbkdf2_hmac(
        'sha256',
        combined,
        salt,
        100000  # iteration count
    )

    return key, salt


# ---------------- AES Encryption ---------------- #
def encrypt_aes(plain_text, key):
    iv = os.urandom(16)

    padder = padding.PKCS7(128).padder()
    padded_data = padder.update(plain_text.encode()) + padder.finalize()

    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()

    encrypted = encryptor.update(padded_data) + encryptor.finalize()

    return base64.b64encode(iv + encrypted).decode()


def decrypt_aes(cipher_text, key):
    raw = base64.b64decode(cipher_text)
    iv = raw[:16]
    encrypted = raw[16:]

    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()

    decrypted_padded = decryptor.update(encrypted) + decryptor.finalize()

    unpadder = padding.PKCS7(128).unpadder()
    decrypted = unpadder.update(decrypted_padded) + unpadder.finalize()

    return decrypted.decode()


# ---------------- Main Program ---------------- #
def main():
    print("=== Secure Cipher + AES (PBKDF2) System ===")

    # Step 1: Input
    text = input("Enter text to encode: ").strip()

    # Step 2: Encode
    encoded = encode_text(text)
    print("\n🔐 Cipher Generated:")
    print(encoded)

    # Step 3: Password + PIN
    password = input("\nEnter 8-character password (can include special characters): ").strip()
    pin = input("Enter 4-digit PIN: ").strip()

    # Validation
    if len(password) != 8:
        print("❌ Password must be exactly 8 characters.")
        return

    if len(pin) != 4 or not pin.isdigit():
        print("❌ PIN must be exactly 4 digits.")
        return

    # Step 4: Derive AES key
    aes_key, salt = derive_aes_key(password, pin)

    print("\n🔑 AES Key (masked):", "*" * 12)

    # Step 5: Encrypt
    encrypted_data = encrypt_aes(encoded, aes_key)

    # Store salt + encrypted data together
    final_output = base64.b64encode(salt).decode() + ":" + encrypted_data

    print("\n🔒 AES Encrypted Cipher:")
    print(final_output)

    # Step 6: Decryption option
    choice = input("\nDo you want to decrypt? (yes/no): ").strip().lower()

    if choice == "yes":
        password_d = input("Enter 8-character password: ").strip()
        pin_d = input("Enter 4-digit PIN: ").strip()

        if password_d != password or pin_d != pin:
            print("❌ Incorrect credentials!")
            return

        try:
            # Extract salt and encrypted data
            salt_b64, encrypted_part = final_output.split(":")
            salt = base64.b64decode(salt_b64)

            # Re-derive same key
            aes_key_d, _ = derive_aes_key(password_d, pin_d, salt)

            # Decrypt
            decrypted_cipher = decrypt_aes(encrypted_part, aes_key_d)

            # Decode custom cipher
            original_text = decode_text(decrypted_cipher)

            print("\n🔓 Decrypted Original Text:")
            print(original_text)

        except Exception:
            print("❌ Decryption failed (wrong key or corrupted data).")

    else:
        print("\nGood luck!")


if __name__ == "__main__":
    main()

=== Secure Cipher + AES (PBKDF2) System ===
Enter text to encode: Let us meet for 2026 Graduate  Award Ceremony.

🔐 Cipher Generated:
♀♣¶ §‼ ♪♣♣¶ ♠☼↕ 2026 •↕☺♦§☺¶♣  ☺↨☺↕♦ ♥♣↕♣♪☼♫↓.

Enter 8-character password (can include special characters): St@ff*rd
Enter 4-digit PIN: 1111

🔑 AES Key (masked): ************

🔒 AES Encrypted Cipher:
IrBt1vE3NB7rCPmd/O8S5Q==:ZppF3MNcdORAtoOSQxx/bRjvzwkQJqA9cpurgHvj+MXQ142wtLS1C50oWoVKkWSETn6bPiEZoKIKuCtPxSIwOl0SYhJuVQcyXyOSwZgpqrSvn+1knAvBBFL6mSVZaD5XWGOSTc3le7vQlvd5M13XvoqQl7B5pMdOWabbACbT0ZI=

Do you want to decrypt? (yes/no): Yes
Enter 8-character password: St@ff*rd
Enter 4-digit PIN: 1111

🔓 Decrypted Original Text:
let us meet for 2026 graduate  award ceremony.
